# Treinamento de modelos

### Disciplina de Sistemas Inteligentes

### Departamento de Informática e Estatística

### Profa. Nathalia da Cruz Alves


**Nota 1**: Em caso de reprodução, publicação ou compartilhamento deste material, a autoria deverá ser obrigatoriamente mantida.

**Nota 2**: Este material foi elaborado com o auxílio de ferramentas de Inteligência Artificial (IA), utilizadas para o refinamento de explicações e revisão de trechos de código. A autoria e a responsabilidade pelo conteúdo final permanecem com a autora.

Estudante(s): insira seu nome aqui


**Observação**: Solicita-se que sejam informados apenas os nomes dos estudantes efetivamente presentes e participantes da atividade.

Nesta aula você vai aprender, na prática, como construir, otimizar e treinar Redes Neurais Convolucionais (CNNs) avançadas para classificação de imagens utilizando o Keras Tuner.

Como objetivos, você verá:

- Configuração de arquiteturas flexíveis para busca automática de hiperparâmetros
- Teste e ajuste de filtros convolucionais, tamanhos de camadas densas e taxas de Dropout
- Otimização da taxa de aprendizado (learning rate) do otimizador Adam
- Aplicação prática de estratégias eficientes como o Random Search
- Integração de aumento de dados (Data Augmentation) direto na pipeline de treino

Neste ambiente Docker/Jupyter, a aula pode ser executada em CPU.
A GPU é opcional e depende de suporte e configuração no host e no container;
não é ativada pelo menu de ambiente de execução do Colab.

Os treinamentos em CPU podem demorar mais. Para conferir se há GPU disponível,
execute a célula de diagnóstico abaixo.


### Configurações iniciais

### Instalação de bibliotecas

A biblioteca `keras-tuner` será utilizada para busca automática de hiperparâmetros.
O pacote `tensorboard` também é necessário para executar as tentativas da busca.
A célula abaixo usa `%pip`, que instala no ambiente Python do kernel deste notebook.
Execute-a antes dos imports. O pacote também está listado no `requirements.txt`
do Docker (`jupyter-notebook/requirements.txt`) para as próximas reconstruções da imagem.


In [1]:
# Instalar no mesmo ambiente Python usado pelo kernel do Jupyter.
%pip install keras-tuner tensorboard


Note: you may need to restart the kernel to use updated packages.


### Import de bibliotecas

A configuração abaixo reduz os logs informativos gerenciados pelo TensorFlow;
avisos e erros continuam visíveis. Algumas mensagens de CUDA/Abseil emitidas
antes da inicialização do logger ainda podem aparecer. A ausência de drivers
CUDA indica execução em CPU e não impede o treinamento nesta aula.

A configuração deve ser definida antes do primeiro import. Se o TensorFlow já
foi carregado, reinicie o kernel e execute as células novamente em ordem.


In [2]:
# Configurar antes do primeiro import do TensorFlow.
# Nível 1: omitir logs INFO, preservando avisos e erros.
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'

# Construção e treinamento das redes neurais
import tensorflow as tf
from tensorflow import keras

# Biblioteca para tuning de hiperparâmetros
import keras_tuner as kt

# Visualização gráfica
import matplotlib.pyplot as plt
import seaborn as sns

# Controle de aleatoriedade
import random

# Manipulação de dados
import numpy as np
import pandas as pd

# Matriz de confusão
from sklearn.metrics import confusion_matrix

I0000 00:00:1793924511.468828    2867 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793924512.750155    2867 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Controle de reprodutibilidade

In [3]:
# Set da seed das bibliotecas para reprodução de resultados
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

### Verificar o dispositivo de execução

Se a lista de GPUs estiver vazia, o TensorFlow usará a CPU e a aula pode continuar.


In [4]:
gpus = tf.config.list_physical_devices('GPU')
print('GPUs disponíveis:', gpus)
print('Execução em GPU.' if gpus else 'Execução em CPU; os treinamentos podem demorar mais.')


GPUs disponíveis: []
Execução em CPU; os treinamentos podem demorar mais.


E0000 00:00:1793924514.284146    2867 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


### Conjunto de dados: Dataset MNIST

O MNIST é um dos datasets mais famosos da área de redes neurais

O dataset MNIST contém:

- 70.000 imagens de dígitos manuscritos
- imagens em escala de cinza
- tamanho 28x28 pixels
- 10 classes (números de 0 a 9 manuscritos)

In [5]:
# Carregando o dataset MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

O MNIST já vem dividido em:

- 60.000 imagens para treinamento
- 10.000 imagens para teste

O conjunto de treino é utilizado para aprender os padrões.

O conjunto de teste é utilizado para verificar a capacidade de generalização do modelo.

In [6]:
print(X_train.shape)  # (60000, 28, 28)
print(X_test.shape)   # (10000, 28, 28)

(60000, 28, 28)
(10000, 28, 28)


#### Normalização dos dados

Redes neurais aprendem melhor quando:

- entradas estão na mesma escala
- números são pequenos
- distribuição é estável


As imagens do Keras MNIST vêm com valores de pixels entre 0 a 255
- 0 = preto
- 255 = branco

Dividindo por 255 os valores ficam entre 0.0 a 1.0

Isso melhora a estabilidade do gradiente e a velocidade de treinamento

In [7]:
X_train = X_train / 255.0
X_test = X_test / 255.0

### Preparação para CNN

As CNNs esperam imagens no formato:
- (amostras, altura, largura, canais)

In [8]:
X_train = X_train.reshape(-1,28,28,1) # O argumento -1 calcula automaticamente o número de imagens.
X_test = X_test.reshape(-1,28,28,1)

print(X_train.shape)

(60000, 28, 28, 1)


### Tuning (ajuste) de Hiperparâmetros

Hiperparâmetros são as configurações definidas manualmente antes de iniciar o treinamento de um modelo de Inteligência Artificial.

Diferente dos parâmetros comuns (como pesos e vieses, que a rede aprende sozinha analisando os dados), os hiperparâmetros controlam a estrutura da rede e a forma como ela aprende.

Exemplos comuns:

- Número de camadas: a profundidade da rede neural.
- Quantidade de neurônios: o tamanho de cada camada (ex: na camada densa final).
- Taxa de aprendizado (Learning Rate): a velocidade com que o modelo atualiza seu conhecimento a cada passo.







**Como funciona o Tuning (Ajuste)?**

O tuning consiste em automatizar a busca pela combinação ideal desses valores para obter o melhor desempenho possível do modelo. Em vez de testar tudo manualmente, utilizam-se algoritmos de busca (como Grid Search, Random Search ou ferramentas como Optuna) para explorar diferentes cenários, como:

Para o tuning, você vai testar automaticamente combinações como:

- Filtros da 1ª camada convolucional (filters_1): testa usar 8, 16 ou 32 filtros para extrair características iniciais da imagem

- Filtros da 2ª camada convolucional (filters_2): testa usar 16, 32 ou 64 filtros para capturar padrões mais complexos

- Quantidade de neurônios na camada densa (dense_units): testa tamanhos que variam de 32 a 128 (com saltos de 32 em 32: 32, 64, 96 ou 128 neurônios)

- Taxa de Dropout (dropout_rate): testa desligar 20%, 30% ou 50% dos neurônios aleatoriamente para evitar que o modelo decore os dados (overfitting)

- Taxa de aprendizado / Learning Rate (learning_rate): testa três velocidades diferentes para o otimizador Adam: 0.01, 0.001 ou 0.0001

In [9]:
def build_model(hp):
  model = keras.Sequential([

      # Camada de entrada (ajustada para imagens 28x28 com 1 canal de cor/escala de cinza)
      keras.layers.Input(shape=(28, 28, 1)),

      # Aumento de dados embutido na pipeline
      keras.Sequential([
          keras.layers.RandomRotation(0.1),
          keras.layers.RandomTranslation(
              height_factor=0.1, width_factor=0.1
          ),
          keras.layers.RandomZoom(0.1),
      ]),

      # Primeira camada convolucional (filtros variáveis)
      tf.keras.layers.Conv2D(
          filters=hp.Choice("filters_1", [8, 16, 32]),
          kernel_size=(3, 3),
          activation="relu",
      ),

      # Normaliza as ativações da camada anterior
      tf.keras.layers.BatchNormalization(),

      tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

      # Segunda camada convolucional (filtros variáveis)
      tf.keras.layers.Conv2D(
          filters=hp.Choice("filters_2", [16, 32, 64]),
          kernel_size=(3, 3),
          activation="relu",
      ),

      # Normaliza as ativações da camada anterior
      tf.keras.layers.BatchNormalization(),

      tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

      tf.keras.layers.Flatten(),

      # Camada densa com quantidade de neurônios variável
      tf.keras.layers.Dense(
          units=hp.Int("dense_units", min_value=32, max_value=128, step=32),
          activation="relu",
      ),

      # Dropout para evitar overfitting com taxa ajustável
      tf.keras.layers.Dropout(rate=hp.Choice("dropout_rate", [0.2, 0.3, 0.5])),

      # Camada de saida
      tf.keras.layers.Dense(10, activation="softmax"),
  ])

  model.compile(
      optimizer=keras.optimizers.Adam(
          learning_rate=hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4])
      ),
      loss="sparse_categorical_crossentropy",
      metrics=["accuracy"],
  )

  return model

#### Random Search

Como as Redes Neurais Convolucionais (CNNs) exigem mais processamento por conta das operações de convolução, o ideal é manter um número prudente de trials (tentativas) e épocas curtas durante a busca para otimizar o tempo.

**Random Search:**
- Seleciona e testa combinações aleatórias de hiperparâmetros dentro dos intervalos definidos
- Treina cada modelo gerado e mede seu desempenho em dados de validação.
- Compara todos os resultados obtidos e escolhe a melhor combinação testada.


OBS: Por que usar Random Search em vez de Grid Search?
O Grid Search tenta testar exaustivamente todas as combinações possíveis, o que gera uma explosão combinatória e torna o processo inviável em termos de tempo e custo computacional. O Random Search, por outro lado, é significativamente mais eficiente e se mostra extremamente eficaz na prática para encontrar ótimos hiperparâmetros com menos esforço.

In [10]:
tuner = kt.RandomSearch(
    # Função de construção de modelos
    build_model,

    # Métrica objetivo
    objective="val_accuracy",

    # Número máximo de testes
    max_trials=5,

    directory="tuner_results_cnn",

    project_name="mnist_cnn_tuning",

    overwrite=True,
)

### Execução da busca

O tempo depende do dispositivo e da configuração testada. Em CPU, a busca de
5 tentativas com 5 épocas por tentativa pode levar bem mais que 5 minutos.
Para uma verificação rápida, reduza temporariamente `max_trials` e `epochs`;
restaure o orçamento antes de comparar os experimentos completos.


In [ ]:
# Rodando a busca (com menos épocas no tuner para agilizar, ex: 5 épocas)
tuner.search(
    X_train,
    y_train,
    epochs=5,
    validation_split=0.2
)

Trial 4 Complete [00h 01m 39s]
val_accuracy: 0.9734166860580444

Best val_accuracy So Far: 0.9757500290870667
Total elapsed time: 00h 06m 20s

Search: Running Trial #5

Value             |Best Value So Far |Hyperparameter
8                 |32                |filters_1
64                |64                |filters_2
32                |64                |dense_units
0.5               |0.3               |dropout_rate
0.01              |0.001             |learning_rate

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.1448 - loss: 2.2445 - val_accuracy: 0.1992 - val_loss: 2.0827
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - accuracy: 0.2134 - loss: 2.0600 - val_accuracy: 0.4552 - val_loss: 1.4651
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - accuracy: 0.3253 - loss: 1.7775 - val_accuracy: 0.4587 - val_loss: 1.4638
Epoch 4/5
 306/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.3813 - loss: 1.6422

In [ ]:
# @title Resultados da busca

trials = list(tuner.oracle.trials.values())

trials = sorted(
    trials,
    key=lambda t: t.score if t.score is not None else float('-inf'),
    reverse=True
)

results = []

for rank, trial in enumerate(trials, start=1):
    row = {
        'score': trial.score
    }

    row.update(trial.hyperparameters.values)

    results.append(row)

df = pd.DataFrame(results)

df

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

best_hps = tuner.get_best_hyperparameters(1)[0]
print(f'Melhor modelo escolhido: {best_hps.values}')

OBS: Se você deseja visualizar os gráficos de treinamento (fit) com fidelidade absoluta aos resultados originais da busca, é necessário rodar o treinamento novamente para as melhores configurações de hiperparâmetros. Para garantir que essas novas execuções reproduzam exatamente o comportamento observado no tuner, é preciso fixar a semente aleatória (seed) antes de reconstruir e treinar cada modelo.

### Exercício 1 - Adição de novos hiperparâmetros

No código da função build_model(hp), atualmente são testadas três taxas de aprendizado (1e-2, 1e-3, 1e-4).

Modifique a definição do otimizador Adam dentro da função para que o Keras Tuner também teste uma taxa de aprendizado menor (1e-5) e utilize a função hp.Choice para dar liberdade ao tuner de escolher entre quatro opções.


### Exercício 2 - Ajuste da camada de Dropout

Crie um novo teste de hiperparâmetro para permitir que o Keras Tuner teste taxas de Dropout mais finas, por exemplo, variando de 0.1 até 0.5 com passos de 0.1 (0.1, 0.2, 0.3, 0.4, 0.5).

Explique em um breve comentário no seu código por que aumentar a taxa de Dropout ajuda a diminuir o overfitting, e por outro lado, por que uma taxa excessiva (ex: 0.8) também não é ideal.

## Avaliação final

Após a seleção do modelo com o melhor desempenho por meio do ajuste de hiperparâmetros, procede-se à etapa de avaliação final.

A avaliação final permite aferir a real capacidade preditiva da inteligência artificial utilizando um conjunto de dados exclusivo para teste.

O conjunto de teste:
- Possui dados inteiramente desconhecidos pelo modelo, pois não foram utilizados em nenhuma fase do treinamento ou da validação
- Por ser composto por amostras inéditas, o conjunto simula o ambiente real, testando a robustez do modelo diante de variações e situações imprevistas encontradas fora do ambiente de desenvolvimento

In [ ]:
test_loss, test_acc = best_model.evaluate(
    X_test,
    y_test
)

# Probabilidades previstas
y_probs = best_model.predict(X_test)

# Classe prevista
y_pred = np.argmax(y_probs, axis=1)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
# @title Visualizar algumas previsões aleatórias
plt.figure(figsize=(12, 4))

for i in range(5):
    idx = np.random.randint(len(X_test))

    plt.subplot(1, 5, i + 1)
    plt.imshow(X_test[idx], cmap='gray')

    pred_label = y_pred[idx]
    true_label = y_test[idx]

    # Verde se acertou
    color = "green" if pred_label == true_label else "red"

    plt.title(f"P: {pred_label} | T: {true_label}", color=color)
    plt.axis('off')

plt.tight_layout()
plt.show()

#### Matriz de confusão

A matriz de confusão mostra:

- Quais classes o modelo acerta
- Quais classes ele confunde


Interpretação da matriz de confusão

- **Diagonal principal**: acertos do modelo

- **Fora da diagonal**: erros de classificação

In [ ]:
# @title Visualizar a matriz de confusão
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão - MNIST")
plt.show()